# Load and visualize five local NomNaOCR sample page triplets

This notebook verifies five image, line-label, and character-detection-label triplets in the persistent lab workspace. It draws line polygons and character-position boxes together. The boxes localize characters but do **not** contain character identities. Start Jupyter from a shell where `LAB_WORKSPACE` is set.

In [ ]:
import os
import socket
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Polygon, Rectangle
from PIL import Image
from nom_ocr_lab import discover_page_pairs, read_detection_annotations, read_line_annotations

job_id = os.environ.get("SLURM_JOB_ID")
print(f"Host: {socket.gethostname()}")
if job_id:
    print(f"Slurm allocation: job {job_id}")
    print(f"CPUs per task: {os.environ.get('SLURM_CPUS_PER_TASK', 'not reported')}")
    print(f"Memory per node: {os.environ.get('SLURM_MEM_PER_NODE', 'not reported')} MB")
    print(f"Visible GPUs: {os.environ.get('CUDA_VISIBLE_DEVICES', 'none requested')}")
else:
    print("WARNING: this kernel is outside a Slurm allocation.")

if "LAB_WORKSPACE" not in os.environ:
    raise RuntimeError("Set LAB_WORKSPACE before starting Jupyter")

SAMPLE_ROOT = Path(os.environ["LAB_WORKSPACE"]) / "datasets" / "nomnaocr-sample"
IMAGE_ROOT = SAMPLE_ROOT / "images"
page_pairs = discover_page_pairs(SAMPLE_ROOT)
assert len(page_pairs) == 5, f"Expected 5 sample triplets, found {len(page_pairs)} in {SAMPLE_ROOT}"

print(f"Found {len(page_pairs)} matched page triplets in {SAMPLE_ROOT}")

In [ ]:
for page_number, pair in enumerate(page_pairs, start=1):
    image = Image.open(pair.image_path).convert("RGB")
    annotations = read_line_annotations(pair.label_path)
    detections = read_detection_annotations(pair.detection_label_path)

    print(f"\nPage {page_number}: {pair.relative_image}")
    print(f"Image size: {image.width} x {image.height} pixels")
    print(f"Loaded {len(annotations)} line labels from: {pair.label_path}")
    print(f"Loaded {len(detections)} character-position boxes from: {pair.detection_label_path}")
    for line_number, annotation in enumerate(annotations, start=1):
        print(f"{line_number:>3}: {annotation.text} | {annotation.points}")

    fig, ax = plt.subplots(figsize=(12, 16))
    ax.imshow(image)
    for line_number, annotation in enumerate(annotations, start=1):
        ax.add_patch(Polygon(annotation.points, closed=True, fill=False, edgecolor="red", linewidth=1.5))
        ax.text(
            annotation.points[0][0], annotation.points[0][1], str(line_number),
            color="yellow", fontsize=8,
            bbox={"facecolor": "black", "alpha": 0.6, "pad": 1},
        )
    for detection in detections:
        left = (detection.x_center - detection.width / 2) * image.width
        top = (detection.y_center - detection.height / 2) * image.height
        ax.add_patch(Rectangle((left, top), detection.width * image.width, detection.height * image.height, fill=False, edgecolor="cyan", linewidth=0.8))
    ax.legend(handles=[Patch(facecolor="none", edgecolor="red", label="Line polygon + transcription"), Patch(facecolor="none", edgecolor="cyan", label="Character position only")], loc="upper right")
    ax.set_title(f"{pair.image_path.name}: {len(annotations)} lines, {len(detections)} character positions")
    ax.axis("off")
    plt.tight_layout()
    plt.show()